# appagentica – RAG sugli algoritmi di ordinamento

Dataset JSON → embedding OpenAI → Qdrant (Docker) → prompt engine.

**Prima di iniziare:** `docker compose up -d` e un file `.env` con `OPENAI_API_KEY`.

In [6]:
# Installazione (una volta sola)
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import json
from pathlib import Path

from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from rich.console import Console
from rich.markdown import Markdown

import config

console = Console()
openai_client = OpenAI(api_key=config.OPENAI_API_KEY)
qdrant = QdrantClient(url=config.QDRANT_URL)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

## 1. Carica il dataset

In [2]:
items = json.loads(Path('data/algorithms.json').read_text(encoding='utf-8'))
print(len(items), 'algoritmi')
items[0]

FileNotFoundError: [Errno 2] No such file or directory: 'data\\algorithms.json'

## 2. Embedding e caricamento in Qdrant

In [3]:
def build_text(i):
    return (f"{i['name']} ({i['language']}). Algoritmo: {i['algorithm']}. {i['description']} "
            f"Quando usarlo: {i['use_when']} Complessita' media {i['time_avg']}, peggiore {i['time_worst']}, "
            f"spazio {i['space']}. Stabile: {'si' if i['stable'] else 'no'}. In place: {'si' if i['in_place'] else 'no'}.")

resp = openai_client.embeddings.create(model=config.EMBEDDING_MODEL, input=[build_text(i) for i in items])
vectors = [d.embedding for d in resp.data]

if qdrant.collection_exists(config.COLLECTION_NAME):
    qdrant.delete_collection(config.COLLECTION_NAME)
qdrant.create_collection(config.COLLECTION_NAME,
    vectors_config=VectorParams(size=config.EMBEDDING_DIM, distance=Distance.COSINE))
qdrant.upsert(config.COLLECTION_NAME,
    points=[PointStruct(id=i['id'], vector=v, payload=i) for i, v in zip(items, vectors)])
print('Caricati', len(items), 'punti')

NameError: name 'openai_client' is not defined

## 3. Prompt engine

In [4]:
SYSTEM_PROMPT = (
    'Sei un assistente esperto di algoritmi di ordinamento. Rispondi in italiano usando SOLO le schede '
    'del contesto. Indica quale algoritmo/funzione consigli, per quale linguaggio, perche (complessita, '
    'stabilita, caso d uso) e mostra un breve esempio di codice. Se nel contesto non c e una risposta adatta, dillo.'
)

def retrieve(question, top_k=5):
    emb = openai_client.embeddings.create(model=config.EMBEDDING_MODEL, input=question)
    return qdrant.query_points(config.COLLECTION_NAME, query=emb.data[0].embedding, limit=top_k, with_payload=True).points

def ask(question):
    points = retrieve(question)
    for p in points:
        print(f"{p.score:.3f}  {p.payload['name']} [{p.payload['language']}]")
    ctx = '\n\n'.join(
        f"- {p.payload['name']} [{p.payload['language']}] | {p.payload['algorithm']} | medio {p.payload['time_avg']}, "
        f"peggiore {p.payload['time_worst']} | stabile: {p.payload['stable']}\n  {p.payload['description']}\n  "
        f"Quando: {p.payload['use_when']}\n  Codice:\n{p.payload['code']}" for p in points)
    out = openai_client.chat.completions.create(model=config.CHAT_MODEL, temperature=0.2, messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Contesto:\n{ctx}\n\nDomanda: {question}'}])
    console.print(Markdown(out.choices[0].message.content))

In [5]:
ask('Devo ordinare 10 milioni di interi in Java su una macchina multi-core')

NameError: name 'openai_client' is not defined

In [ ]:
ask('Mi servono solo i 10 elementi piu grandi di una lista enorme in Python')